# Audio Editor
## This notebook outlines the techniques used to edit the audio signal (removing long silences)

### Import the libraries

In [37]:
#!pip install pydub

In [39]:
from pydub import AudioSegment
import numpy as np

### Read MP3 file using PyDub

In [42]:
audiofile = AudioSegment.from_file("obama.mp3")

In [43]:
audiofile

### Convert the samples into Numpy array

In [47]:
data_mp3 = np.array(audiofile.get_array_of_samples())

In [49]:
data_mp3.shape

(2985984,)

### Get the frame rate

In [52]:
fs_mp3 = audiofile.frame_rate

In [54]:
fs_mp3

48000

### Find the signal duration

In [57]:
print(f'Signal Duration = {data_mp3.shape[0] / (2 * fs_mp3)} seconds')

Signal Duration = 31.104 seconds


### Read a wav file
- Use scipy's wavfile

In [64]:
from scipy.io import wavfile
fs_wav, data_wav = wavfile.read("test.wav")

### Get the frame rate

In [67]:
fs_wav

48000

In [69]:
data_wav.shape

(48000,)

### Get the duration of the signal

In [72]:
print(f'Signal Duration = {data_wav.shape[0] / fs_wav} seconds')

Signal Duration = 1.0 seconds


### Normalize the signal

In [75]:
data_wav_norm = data_wav / (2**15)

### Length of the signal

In [78]:
signal_len = len(data_wav_norm)
signal_len

48000

### Fix segment size in seconds

In [81]:
segment_size_t = 1 

### Segment size in samples

In [84]:
segment_size = segment_size_t * fs_wav

### Split audio signal into 1 second segments

In [87]:
segments = np.array([data_wav_norm[x:x + segment_size] for x in
                     np.arange(0, signal_len, segment_size)])

### How many segments ?

In [90]:
segments.shape[0]

1

### Save each segment in a seperate filename

In [93]:
for iS, s in enumerate(segments):
    wavfile.write("obama_segment_{0:d}_{1:d}.wav".format(segment_size_t * iS,
                                                              segment_size_t * (iS + 1)), fs_wav, (s))

### Remove pauses using an energy threshold = 50% of the median energy

In [96]:
energies = [(s**2).sum() / len(s) for s in segments]

### Fix 50% as threshold

In [99]:
thres = 0.5 * np.median(energies)

### Collect the indexes of segments which is above this 50% threshold

In [102]:
index_of_segments_to_keep = (np.where(energies > thres)[0])

### Get segments that have energies higher than the threshold

In [105]:
segments2 = segments[index_of_segments_to_keep]

### Concatenate segments to signal

In [108]:
new_signal = np.concatenate(segments2)

### Write the file

In [111]:
wavfile.write("obama_processed.wav", fs_wav, new_signal)

## ASSIGNMENT: Take any mp3 file containing speeches of someone of your choice with pauses inbetween the speech and apply the above techniques to remove pauses. Become an audio editor!
- Submit the mp3 file of original speech, notebook file, mp3 file of edited speech

In [120]:
from pydub import AudioSegment
import numpy as np
from scipy.io import wavfile
import os

# === Step 1: 讀取 MP3 檔並轉換為 WAV ===
mp3_path = "obama.mp3"
wav_path = "obama-speech.wav"

audio = AudioSegment.from_mp3(mp3_path)
audio = audio.set_channels(1)        # 強制單聲道
audio = audio.set_frame_rate(44100)  # 設定取樣率
audio.export(wav_path, format="wav")

# === Step 2: 使用 scipy 讀取 WAV 資料 ===
fs_wav, data_wav = wavfile.read(wav_path)

# === Step 3: 對資料做正規化，轉為 float，避免溢位問題 ===
data_wav_norm = data_wav.astype(np.float32)

# === Step 4: 將音訊每 1 秒切一段 ===
segment_size_t = 1  # 1 秒
segment_size = int(fs_wav * segment_size_t)
signal_len = len(data_wav_norm)

segments = []
for start in range(0, signal_len, segment_size):
    end = start + segment_size
    segment = data_wav_norm[start:end]
    if len(segment) == segment_size:  # 忽略最後不足 1 秒的段落
        segments.append(segment)

# === Step 5: 判斷能量，過濾靜音段落 ===
threshold = 500  # 根據音訊大小調整門檻
segments_to_keep = []

for segment in segments:
    energy = np.mean(np.abs(segment))
    if energy > threshold:
        segments_to_keep.append(segment)

# === Step 6: 合併非靜音片段，輸出新的音訊 ===
if len(segments_to_keep) == 0:
    print("No segments above threshold. Adjust the threshold value.")
else:
    new_signal = np.concatenate(segments_to_keep)
    output_path = "speech_cleaned.wav"
    wavfile.write(output_path, fs_wav, new_signal.astype(np.int16))
    print(f"✅ 處理完成：{output_path}")


✅ 處理完成：speech_cleaned.wav
